# Export Lumis-1 Speaker Model to LiteRT (Colab GPU Edition)

This notebook exports the fine-tuned `lumis1-speaker-v1` (Granite 4.0 1B Dense) model to `.litertlm` (TensorFlow Lite) format.

**Optimization:** Uses GPU (CUDA) and FP16 to reduce System RAM usage (OOM prevention).
**Target:** LiteRT / MediaPipe LLM Inference
**Quantization:** INT8 Dynamic (Robust) or INT4 (Experimental)

**Prerequisites:**
1. Runtime Type: **T4 GPU** (Required).
2. **Upload `lumis1-speaker-v1.zip` to your Google Drive (Root Folder).**
3. Run all cells.

In [ ]:
# 1. Install Dependencies (Nightly Stack)
# Uninstall potentially conflicting stable versions
!pip uninstall -y tensorflow tensorflow-gpu tensorflow-cpu tf-nightly

# Install Nightly Stack (AI Edge Torch Nightly usually requires TF Nightly)
!pip install ai-edge-torch-nightly tf-nightly transformers torch sentencepiece accelerate

In [ ]:
import os
import torch
import tensorflow as tf
import ai_edge_torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import shutil
import zipfile

print(f"PyTorch: {torch.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"AI Edge Torch: {ai_edge_torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

In [ ]:
# 2. Configure Paths & Handle Uploads
MODEL_DIR_NAME = "lumis1-speaker-v1"
MODEL_ZIP_NAME = "lumis1-speaker-v1.zip"
EXPORT_DIR_NAME = "lumis1-speaker-v1-litert"

# --- MOUNT GOOGLE DRIVE ---
from google.colab import drive
print("Mounting Google Drive... (Follow the popup instructions if prompted)")
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive"
BASE_PATH = "/content"

# Look for Zip in Drive
DRIVE_ZIP_PATH = os.path.join(DRIVE_ROOT, MODEL_ZIP_NAME)
LOCAL_ZIP_PATH = os.path.join(BASE_PATH, MODEL_ZIP_NAME)
MODEL_PATH = os.path.join(BASE_PATH, MODEL_DIR_NAME)
EXPORT_DIR = os.path.join(BASE_PATH, EXPORT_DIR_NAME)

# Copy Zip from Drive to Colab VM (Faster IO after copy)
if os.path.exists(DRIVE_ZIP_PATH) and not os.path.exists(LOCAL_ZIP_PATH):
    print(f"Found {MODEL_ZIP_NAME} in Drive! Copying to local VM for speed...")
    shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
    print("Copy complete.")
elif not os.path.exists(LOCAL_ZIP_PATH) and not os.path.exists(MODEL_PATH):
    print(f"ERROR: Could not find {MODEL_ZIP_NAME} in Google Drive.")
    print("Please upload 'lumis1-speaker-v1.zip' to the 'My Drive' root in your Google Drive.")
    raise FileNotFoundError("Zip file missing from Google Drive.")

# Extract
if os.path.exists(LOCAL_ZIP_PATH) and not os.path.exists(MODEL_PATH):
    print(f"Extracting {LOCAL_ZIP_PATH}...")
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(BASE_PATH)
    print("Extraction complete.")

In [ ]:
# 3. Load Model (On GPU + FP16)
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}. Please check upload.")

print(f"Loading model from: {MODEL_PATH}")

# Optimize: Use Float16 and CUDA to save System RAM
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.float16, # FP16 saves 50% RAM compared to FP32
    device_map=device,
    trust_remote_code=True
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("Model loaded on GPU.")

In [ ]:
# 4. Monkey Patch (Fix Dynamo/Granite Compatibility)
# Critical for GraniteMoeHybrid / Dense architectures on ai-edge-torch
print("Patching _update_mamba_mask to avoid Dynamo graph break...")

def safe_update_mamba_mask(self, attention_mask, cache_position):
    # Bypass logic for export (safe for Dense/No-Mamba models)
    return None 

# Apply patch
if hasattr(model, 'model') and hasattr(model.model, '_update_mamba_mask'):
    import types
    model.model._update_mamba_mask = types.MethodType(safe_update_mamba_mask, model.model)
    print("Patched model.model._update_mamba_mask")
elif hasattr(model, '_update_mamba_mask'):
    import types
    model._update_mamba_mask = types.MethodType(safe_update_mamba_mask, model)
    print("Patched model._update_mamba_mask")
else:
    print("WARNING: Could not find _update_mamba_mask to patch. Export might fail.")

In [ ]:
# 5. Configure Quantization (Robust Import)
try:
    from ai_edge_torch.generative.quantize import quant_recipe as quant_recipes
    print("Imported quant_recipe from ai_edge_torch.generative.quantize")
except ImportError:
    try:
        from ai_edge_torch.quantize import quant_recipes
        print("Imported quant_recipes from ai_edge_torch.quantize")
    except ImportError:
        quant_recipes = None
        print("CRITICAL ERROR: Could not import quantization recipes.")

# Select INT8 Dynamic for stability
quant_config = None
if quant_recipes and hasattr(quant_recipes, 'full_int8_dynamic_quantization'):
    print("Using 'full_int8_dynamic_quantization' (Recommended).")
    quant_config = quant_recipes.full_int8_dynamic_quantization()
elif quant_recipes and hasattr(quant_recipes, 'generative_int4_quantization'):
    print("WARNING: INT8 recipe not found. Using INT4 (Experimental).")
    quant_config = quant_recipes.generative_int4_quantization()
else:
    print("CRITICAL: No valid quantization recipe found.")

In [ ]:
# 6. Export
os.makedirs(EXPORT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(EXPORT_DIR, "lumis1-speaker-v1-int8.litertlm")

sample_text = "Hello, how are you?"
# CRITICAL: Move sample inputs to SAME DEVICE as Model (GPU)
sample_input_ids = tokenizer(sample_text, return_tensors="pt").input_ids.to(device)
sample_args = (sample_input_ids,)

print(f"Starting Conversion on {device} (this may take 5-10 mins)...")
if quant_config:
    # Note: ai_edge_torch might auto-move to CPU for tracing, but starting on GPU saves loading RAM
    edge_model = ai_edge_torch.convert(model, sample_args, quant_config=quant_config)
    edge_model.export(OUTPUT_FILE)
    print(f"SUCCESS: Model exported to {OUTPUT_FILE}")
else:
    print("Export skipped due to missing config.")

In [ ]:
# 7. Copy Tokenizer Resources
for f in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "vocab.json", "merges.txt"]:
    src = os.path.join(MODEL_PATH, f)
    dst = os.path.join(EXPORT_DIR, f)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {f}")

In [ ]:
# 8. Save to Drive (Result)
RESULT_ZIP = os.path.join(DRIVE_ROOT, "lumis1-speaker-v1-litert.zip")
print(f"Zipping result to {RESULT_ZIP}...")
shutil.make_archive(RESULT_ZIP.replace(".zip", ""), 'zip', EXPORT_DIR)
print("Done! Check your Google Drive for the result.")